# Heart Disease Prediction System

Comparative machine learning analysis using three standardized clinical datasets.

**Dataset size:** 1,573 records  
**Task:** Binary heart disease classification + exploratory K-Modes clustering


## 1. Install and Import Dependencies


In [ ]:
%pip install kmodes

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from kmodes.kmodes import KModes
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
            import warnings
warnings.filterwarnings('ignore')


## 2. Load, Standardize and Merge the Datasets

Place the three CSV files inside the `data/` directory when running from the repository. In Google Colab, upload/mount the files and update `DATA_DIR` if required.


In [ ]:
from pathlib import Path

DATA_DIR = Path('data')
if not DATA_DIR.exists() and Path('/content').exists():
    DATA_DIR = Path('/content')
print(f'Using dataset directory: {DATA_DIR.resolve()}')

def load_and_merge_data(file_paths):
    df2 = pd.read_csv(DATA_DIR / 'dataset_2.csv')
    df3 = pd.read_csv(DATA_DIR / 'dataset_3.csv')
    df1 = pd.read_csv(DATA_DIR / 'dataset_1.csv')

    column_mapping = {
        'Age': 'age', 'S': 'sex', 'Cholesterol': 'chol',
        'Blood Pressure': 'trestbps', 'Heart Rate': 'thalach',
        'Exercise Induced Angina': 'exang', 'Heart Disease': 'target'
    }
    df1.rename(columns=column_mapping, inplace=True)

    cp_mapping = {
        'Typical Angina': 0,
        'Atypical Angina': 1,
        '0n-anginal Pain': 2,
        'Asymptomatic': 3
    }
    df1['cp'] = df1['Chest Pain Type'].map(cp_mapping)
    df1['fbs'] = (df1['Blood Sugar'] > 120).astype(int)

    standard_cols = df2.columns
    df1 = df1.reindex(columns=standard_cols)

    merged_df = pd.concat([df1, df2, df3], ignore_index=True)
    merged_df.replace('?', np.nan, inplace=True)
    for col in ['ca', 'thal']:
        merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

    print('Dataset shapes:', df1.shape, df2.shape, df3.shape)
    print('Merged shape:', merged_df.shape)
    return merged_df


## 3. Preprocessing


In [ ]:
def preprocess_data(df):
    num_cols = df.select_dtypes(include=np.number).columns
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())

    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    for col in df.select_dtypes(include=['object']).columns:
        df[col] = LabelEncoder().fit_transform(df[col])

    X = df.drop('target', axis=1)
    y = df['target']
    return X, y, df


## 4. Exploratory Data Analysis


In [ ]:
def exploratory_data_analysis(df):
    plt.figure(figsize=(12, 9))
    sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Matrix of Features')
    plt.show()

    plt.figure(figsize=(7, 5))
    sns.countplot(x='target', data=df)
    plt.title('Distribution of Heart Disease')
    plt.xlabel('Heart Disease (0 = No, 1 = Yes)')
    plt.ylabel('Count')
    plt.show()


## 5. Train-Test Split, Feature Selection and Scaling


In [ ]:
merged_df = load_and_merge_data(['dataset_1.csv', 'dataset_2.csv', 'dataset_3.csv'])
X, y, preprocessed_df = preprocess_data(merged_df.copy())
exploratory_data_analysis(preprocessed_df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

selector = SelectKBest(score_func=chi2, k=10)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)
selected_features = X.columns[selector.get_support()]
print('Selected features:', list(selected_features))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)


## 6. Model Training and Evaluation


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Support Vector Machine': SVC(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Multilayer Perceptron': MLPClassifier(random_state=42, max_iter=500),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

model_results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    model_results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'report': classification_report(y_test, y_pred, output_dict=True),
        'confusion_matrix': confusion_matrix(y_test, y_pred)
    }
    print(f'\n{name}')
    print(classification_report(y_test, y_pred))
    sns.heatmap(model_results[name]['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Disease', 'Disease'], yticklabels=['No Disease', 'Disease'])
    plt.title(f'Confusion Matrix for {name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()


## 7. K-Modes Clustering


In [ ]:
df_cluster = preprocessed_df.drop('target', axis=1)
cost = []
for k in range(1, 6):
    kmode = KModes(n_clusters=k, init='random', n_init=5, verbose=0, random_state=42)
    kmode.fit_predict(df_cluster)
    cost.append(kmode.cost_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 6), cost, 'bx-')
plt.xlabel('No. of clusters')
plt.ylabel('Cost')
plt.title('Elbow Method For Optimal k')
plt.show()

k_optimal = 2
km = KModes(n_clusters=k_optimal, init='random', n_init=5, verbose=0, random_state=42)
clusters = km.fit_predict(df_cluster)
preprocessed_df['cluster'] = clusters
print(preprocessed_df.groupby(['cluster', 'target']).size())


## 8. Conclusion

In the executed experiment, Random Forest achieved the highest test-set accuracy among the evaluated supervised models, while K-Modes provided an exploratory unsupervised view of patient-group structure.
